# Phase 12 — Final Evaluation, Threshold Optimization, Error Analysis & Explainability

## Objective

This is the final model evaluation stage.

We will:
1. Load the tuned model from Phase 11.
2. Evaluate it on the **untouched Phase 5 validation set**.
3. Select the operating classification threshold using validation data only.
4. Evaluate exactly once on the **untouched test set**.
5. Compare the final model against the Phase 7 baseline.
6. Analyze false positives and false negatives.
7. Study performance across customer segments.
8. Inspect feature importance / model explainability.
9. Quantify the final improvement.
10. Freeze the final model configuration for Phase 13.

### Critical rule

The test set must not influence:
- hyperparameter tuning,
- threshold selection,
- model selection,
- feature selection,
- error-analysis-driven model changes.

The test set is the final unbiased holdout.

## Project evaluation hierarchy

### Primary metric
**PR-AUC / Average Precision**

Useful when the positive class is relatively less frequent.

### Secondary metrics
- ROC-AUC
- Precision
- Recall
- F1
- Accuracy
- Confusion matrix

### Business interpretation

The model predicts:

> Will this customer make at least one purchase during the next 30 days?

A false positive means we target a customer who does not purchase.

A false negative means we miss a customer who would purchase.

Therefore, the appropriate threshold depends on the cost of these two errors.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

for p in [MODELS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)

## 1. Load Phase 5 validation/test datasets

These datasets contain the temporal train/validation/test split created in Phase 5.

We will **not reconstruct the split** here.

In [ ]:
X_val = pd.read_parquet(PROCESSED_DIR / "validation_phase5.parquet")
X_test = pd.read_parquet(PROCESSED_DIR / "test_phase5.parquet")

y_val = pd.read_parquet(
    PROCESSED_DIR / "y_validation_phase5.parquet"
).squeeze()

y_test = pd.read_parquet(
    PROCESSED_DIR / "y_test_phase5.parquet"
).squeeze()

print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nValidation positive rate:", round(y_val.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

## 2. Load the frozen Phase 11 model

The model configuration was selected using temporal CV.

We do not tune anything in this notebook.

In [ ]:
with open(
    RESULTS_DIR / "phase11_best_params.json",
    "r",
    encoding="utf-8"
) as f:
    phase11_config = json.load(f)

best_model_name = phase11_config["model"]

model_path = (
    MODELS_DIR /
    (best_model_name.lower().replace(" ", "_") + "_phase11_tuned.joblib")
)

preprocessor_path = MODELS_DIR / "preprocessor_phase11_tuned.joblib"

final_model = joblib.load(model_path)
final_preprocessor = joblib.load(preprocessor_path)

print("Model:", best_model_name)
print("Model file:", model_path)

## 3. Prepare validation and test matrices

`customer_id` is an identifier, not an ML feature.

No images are used.

In [ ]:
X_val_ml = X_val.drop(columns=["customer_id"], errors="ignore")
X_test_ml = X_test.drop(columns=["customer_id"], errors="ignore")

X_val_t = final_preprocessor.transform(X_val_ml)
X_test_t = final_preprocessor.transform(X_test_ml)

print("Validation transformed:", X_val_t.shape)
print("Test transformed:", X_test_t.shape)

## 4. Generate probability predictions

We use probabilities rather than immediately converting them to 0/1.

Why?

Because the default threshold of 0.5 is not necessarily optimal for an imbalanced classification problem.

In [ ]:
val_proba = final_model.predict_proba(X_val_t)[:, 1]
test_proba = final_model.predict_proba(X_test_t)[:, 1]

print("Validation probability range:",
      round(val_proba.min(), 4), "to", round(val_proba.max(), 4))

print("Test probability range:",
      round(test_proba.min(), 4), "to", round(test_proba.max(), 4))

## 5. Validation threshold optimization

We search thresholds on validation data only.

The test set is not involved.

We evaluate:
- precision
- recall
- F1
- accuracy

The default threshold is 0.50, but the best F1 threshold can be substantially different.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

thresholds = np.arange(0.05, 0.96, 0.01)

threshold_rows = []

for threshold in thresholds:
    pred = (val_proba >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "accuracy": accuracy_score(y_val, pred),
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
    })

threshold_results = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

best_threshold = float(best_threshold_row["threshold"])

print("Best validation F1 threshold:", best_threshold)
display(best_threshold_row.to_frame().T)

## 6. Visualize threshold trade-offs

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    label="Precision"
)
plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    label="Recall"
)
plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    label="F1"
)
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Validation Threshold Trade-off")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 7. Validation performance at the selected threshold

In [ ]:
val_pred = (val_proba >= best_threshold).astype(int)

val_metrics = {
    "accuracy": accuracy_score(y_val, val_pred),
    "precision": precision_score(y_val, val_pred, zero_division=0),
    "recall": recall_score(y_val, val_pred, zero_division=0),
    "f1": f1_score(y_val, val_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_val, val_proba),
    "pr_auc": average_precision_score(y_val, val_proba),
}

val_metrics_df = pd.DataFrame([val_metrics])

display(val_metrics_df)

## 8. Final test evaluation

### Important

This is the first time in the project where the final tuned model is evaluated on the Phase 5 test set.

The threshold was selected using validation data.

The test set is therefore an unbiased estimate of final generalization performance.

In [ ]:
test_pred = (test_proba >= best_threshold).astype(int)

test_metrics = {
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "pr_auc": average_precision_score(y_test, test_proba),
}

test_metrics_df = pd.DataFrame([test_metrics])

display(test_metrics_df)

In [ ]:
print("TEST CLASSIFICATION REPORT")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["No Purchase", "Purchase"],
        zero_division=0
    )
)

## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, test_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual: No Purchase", "Actual: Purchase"],
    columns=["Predicted: No Purchase", "Predicted: Purchase"]
)

display(cm_df)

## 10. ROC and Precision-Recall curves

ROC-AUC measures ranking quality across thresholds.

PR-AUC is particularly useful here because the positive class can be imbalanced.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fpr, tpr, _ = roc_curve(y_test, test_proba)
precision_curve, recall_curve, _ = precision_recall_curve(
    y_test, test_proba
)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f"ROC-AUC = {test_metrics['roc_auc']:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
plt.plot(
    recall_curve,
    precision_curve,
    label=f"PR-AUC = {test_metrics['pr_auc']:.4f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Test Precision-Recall Curve")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 11. Compare default threshold vs optimized threshold

Threshold optimization changes the classification decision, but not the probability-ranking metrics such as ROC-AUC and PR-AUC.

In [ ]:
default_test_pred = (test_proba >= 0.50).astype(int)

threshold_comparison = pd.DataFrame([
    {
        "threshold": 0.50,
        "accuracy": accuracy_score(y_test, default_test_pred),
        "precision": precision_score(y_test, default_test_pred, zero_division=0),
        "recall": recall_score(y_test, default_test_pred, zero_division=0),
        "f1": f1_score(y_test, default_test_pred, zero_division=0),
    },
    {
        "threshold": best_threshold,
        "accuracy": accuracy_score(y_test, test_pred),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
    }
])

display(threshold_comparison)

## 12. Compare final model against the Phase 7 baseline

The baseline is Logistic Regression from Phase 7.

We load its saved test results rather than retraining it.

In [ ]:
baseline_file = RESULTS_DIR / "phase7_test_baseline_results.csv"

if baseline_file.exists():
    baseline_results = pd.read_csv(baseline_file)
    display(baseline_results)
else:
    baseline_results = pd.DataFrame()
    print("Phase 7 baseline result file was not found.")

In [ ]:
baseline_pr_auc = np.nan
baseline_roc_auc = np.nan
baseline_f1 = np.nan

if not baseline_results.empty:
    cols = {c.lower(): c for c in baseline_results.columns}

    for key in ["pr_auc", "roc_auc", "f1"]:
        if key in cols:
            value = pd.to_numeric(
                baseline_results.iloc[0][cols[key]],
                errors="coerce"
            )
            if key == "pr_auc":
                baseline_pr_auc = value
            elif key == "roc_auc":
                baseline_roc_auc = value
            elif key == "f1":
                baseline_f1 = value

comparison = pd.DataFrame([
    {
        "model": "Phase 7 Logistic Regression",
        "PR-AUC": baseline_pr_auc,
        "ROC-AUC": baseline_roc_auc,
        "F1": baseline_f1,
    },
    {
        "model": f"Phase 11 Tuned {best_model_name}",
        "PR-AUC": test_metrics["pr_auc"],
        "ROC-AUC": test_metrics["roc_auc"],
        "F1": test_metrics["f1"],
    }
])

display(comparison)

## 13. Quantify improvement

For placement reporting, distinguish carefully between:

### Absolute improvement
```text
New score − Baseline score
```

### Relative improvement
```text
(New − Baseline) / Baseline × 100
```

Do not call relative improvement "accuracy improvement" unless accuracy itself is the metric.

In [ ]:
improvement_rows = []

for metric in ["PR-AUC", "ROC-AUC", "F1"]:
    baseline_value = comparison.loc[
        comparison.model == "Phase 7 Logistic Regression", metric
    ].iloc[0]

    final_value = comparison.loc[
        comparison.model.str.startswith("Phase 11 Tuned"), metric
    ].iloc[0]

    if pd.notna(baseline_value) and baseline_value != 0:
        relative = (final_value - baseline_value) / baseline_value * 100
    else:
        relative = np.nan

    improvement_rows.append({
        "metric": metric,
        "baseline": baseline_value,
        "final": final_value,
        "absolute_improvement": final_value - baseline_value,
        "relative_improvement_percent": relative,
    })

improvement_df = pd.DataFrame(improvement_rows)
display(improvement_df)

## 14. Error analysis dataset

We now attach:
- actual target
- predicted probability
- prediction
- error type

Error categories:

- True Positive
- True Negative
- False Positive
- False Negative

In [ ]:
error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted_probability"] = test_proba
error_df["predicted"] = test_pred

def error_type(row):
    if row.actual == 1 and row.predicted == 1:
        return "True Positive"
    if row.actual == 0 and row.predicted == 0:
        return "True Negative"
    if row.actual == 0 and row.predicted == 1:
        return "False Positive"
    return "False Negative"

error_df["error_type"] = error_df.apply(error_type, axis=1)

display(
    error_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="count")
)

## 15. False-positive and false-negative profiles

We compare the major behavioral features of the error groups.

This can reveal where the model struggles.

In [ ]:
numeric_error_cols = [
    c for c in [
        "recency_days",
        "total_items",
        "purchase_days",
        "unique_articles",
        "total_spend",
        "avg_price",
        "customer_tenure_days",
        "purchase_rate",
        "recent_30d_items",
        "recent_30d_spend",
        "recent_90d_items",
        "recent_90d_spend",
    ]
    if c in error_df.columns
]

error_profile = (
    error_df.groupby("error_type")[numeric_error_cols]
    .median()
    .T
)

display(error_profile)

## 16. Probability distribution by error type

False positives and false negatives are especially informative.

- False positives: model was confident but customer did not purchase.
- False negatives: customer purchased but model gave insufficient probability.

In [ ]:
plt.figure(figsize=(9, 5))

for label in [
    "True Positive",
    "False Positive",
    "False Negative",
    "True Negative",
]:
    subset = error_df.loc[
        error_df.error_type == label,
        "predicted_probability"
    ]

    if len(subset):
        plt.hist(
            subset,
            bins=30,
            alpha=0.45,
            label=label
        )

plt.axvline(
    best_threshold,
    linestyle="--",
    label=f"Threshold = {best_threshold:.2f}"
)

plt.xlabel("Predicted Purchase Probability")
plt.ylabel("Number of Customers")
plt.title("Prediction Probability by Error Type")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 17. Segment-level performance

Overall metrics can hide weak performance for particular customer groups.

We evaluate performance across:
- age bands
- club membership
- recent activity
- customer engagement

These are diagnostic analyses, not additional model tuning.

In [ ]:
segment_df = X_test.copy()
segment_df["actual"] = y_test.values
segment_df["proba"] = test_proba
segment_df["predicted"] = test_pred

if "age" in segment_df.columns:
    segment_df["age_band"] = pd.cut(
        segment_df["age"],
        bins=[0, 18, 25, 35, 45, 55, 100],
        labels=["<=18", "19-25", "26-35", "36-45", "46-55", "56+"],
        include_lowest=True
    )

if "recent_30d_items" in segment_df.columns:
    segment_df["recent_activity_band"] = pd.cut(
        segment_df["recent_30d_items"],
        bins=[-1, 0, 1, 3, 10, np.inf],
        labels=["0", "1", "2-3", "4-10", "10+"]
    )

def segment_metrics(group):
    if len(group) < 30 or group.actual.nunique() < 2:
        return pd.Series({
            "customers": len(group),
            "positive_rate": group.actual.mean(),
            "pr_auc": np.nan,
            "roc_auc": np.nan,
            "f1": np.nan,
            "precision": np.nan,
            "recall": np.nan,
        })

    return pd.Series({
        "customers": len(group),
        "positive_rate": group.actual.mean(),
        "pr_auc": average_precision_score(group.actual, group.proba),
        "roc_auc": roc_auc_score(group.actual, group.proba),
        "f1": f1_score(group.actual, group.predicted, zero_division=0),
        "precision": precision_score(group.actual, group.predicted, zero_division=0),
        "recall": recall_score(group.actual, group.predicted, zero_division=0),
    })

In [ ]:
if "age_band" in segment_df.columns:
    age_performance = (
        segment_df.groupby("age_band", observed=False)
        .apply(segment_metrics)
        .reset_index()
    )
    display(age_performance)

In [ ]:
if "recent_activity_band" in segment_df.columns:
    activity_performance = (
        segment_df.groupby("recent_activity_band", observed=False)
        .apply(segment_metrics)
        .reset_index()
    )
    display(activity_performance)

## 18. Feature importance / explainability

For tree-based models, feature importance provides a global explanation of which transformed features contribute most to predictions.

If the final model exposes `feature_importances_`, we map those values back to the preprocessing output features.

For XGBoost and tree ensembles this is especially useful for interpreting the final model.

In [ ]:
feature_names = final_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame()

if hasattr(final_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": final_model.feature_importances_
    }).sort_values("importance", ascending=False)

elif hasattr(final_model, "coef_"):
    coef = np.asarray(final_model.coef_).ravel()
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": np.abs(coef)
    }).sort_values("importance", ascending=False)

if not importance_df.empty:
    display(importance_df.head(30))
else:
    print("Feature importance is not directly available for this model.")

## 19. Plot top global feature importance

In [ ]:
if not importance_df.empty:
    top = importance_df.head(20).sort_values("importance")

    plt.figure(figsize=(9, 7))
    plt.barh(top["feature"], top["importance"])
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.title("Top 20 Global Feature Importances")
    plt.grid(axis="x", alpha=0.2)
    plt.show()

## 20. Optional permutation importance on validation data

Permutation importance measures how much validation performance drops when a feature is shuffled.

Because the dataset can be large, this is performed on a sample.

This is a model-agnostic explainability check.

In [ ]:
from sklearn.inspection import permutation_importance

MAX_PERM_ROWS = 10_000

rng = np.random.RandomState(RANDOM_STATE)

if len(X_val_ml) > MAX_PERM_ROWS:
    perm_idx = rng.choice(len(X_val_ml), MAX_PERM_ROWS, replace=False)
else:
    perm_idx = np.arange(len(X_val_ml))

X_perm = X_val_t[perm_idx]
y_perm = y_val.iloc[perm_idx]

try:
    perm = permutation_importance(
        final_model,
        X_perm,
        y_perm,
        scoring="average_precision",
        n_repeats=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    perm_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    }).sort_values("importance_mean", ascending=False)

    display(perm_df.head(20))
except Exception as e:
    perm_df = pd.DataFrame()
    print("Permutation importance could not be computed:", e)

## 21. Save evaluation artifacts

These outputs will be used for the final project report and CV metrics.

In [ ]:
threshold_results.to_csv(
    RESULTS_DIR / "phase12_threshold_analysis.csv",
    index=False
)

val_metrics_df.to_csv(
    RESULTS_DIR / "phase12_validation_metrics.csv",
    index=False
)

test_metrics_df.to_csv(
    RESULTS_DIR / "phase12_test_metrics.csv",
    index=False
)

comparison.to_csv(
    RESULTS_DIR / "phase12_model_comparison.csv",
    index=False
)

improvement_df.to_csv(
    RESULTS_DIR / "phase12_improvement.csv",
    index=False
)

cm_df.to_csv(
    RESULTS_DIR / "phase12_confusion_matrix.csv"
)

error_df.to_parquet(
    RESULTS_DIR / "phase12_test_error_analysis.parquet",
    index=False
)

if not importance_df.empty:
    importance_df.to_csv(
        RESULTS_DIR / "phase12_feature_importance.csv",
        index=False
    )

if not perm_df.empty:
    perm_df.to_csv(
        RESULTS_DIR / "phase12_permutation_importance.csv",
        index=False
)

final_evaluation_config = {
    "phase": 12,
    "model": best_model_name,
    "threshold": best_threshold,
    "threshold_selection_data": "validation_only",
    "test_used_for_threshold_selection": False,
    "test_used_for_model_selection": False,
    "test_metrics": test_metrics,
}

with open(
    RESULTS_DIR / "phase12_final_evaluation_config.json",
    "w"
) as f:
    json.dump(final_evaluation_config, f, indent=2)

print("Phase 12 artifacts saved.")

# Phase 12 — Final Conclusions

### 1. Model quality
The final model is judged primarily using **test PR-AUC**, with ROC-AUC, F1, precision, recall and accuracy as supporting metrics.

### 2. Threshold
The classification threshold is selected using the validation set only.

### 3. Generalization
The test set provides the final estimate of unseen-customer/time-period performance.

### 4. Error analysis
False positives and false negatives reveal where the model is making mistakes and which customer behaviors are difficult to predict.

### 5. Explainability
Global feature importance and permutation importance provide evidence for which behavioral and demographic variables drive predictions.

### 6. Placement-ready statement

> **Evaluated the tuned temporal ML model on a strict holdout test period, optimized the operating threshold using validation data, performed segment-wise error analysis, and interpreted model predictions using global and permutation-based feature importance.**

## Next phase

**Phase 13 — Final Model Packaging, Inference Pipeline & Deployment**

We will turn the trained model into a reusable prediction pipeline:
- saved preprocessing + model,
- input schema,
- inference function,
- batch prediction,
- probability + classification output,
- model versioning,
- deployment-ready structure,
- and final project architecture.